In [1]:
import torch
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

2.13.0+cu130
13.0
True
NVIDIA GeForce RTX 3060 Laptop GPU


In [2]:
import json
import random
import sys
from pathlib import Path
import numpy as np

current_dir = Path.cwd().resolve()
WORKSPACE_ROOT = next(
    (candidate for candidate in (current_dir, *current_dir.parents)
     if (candidate / "Temporal-Testing").is_dir() and (candidate / "Fin-RATE").is_dir()),
    None,
)
if WORKSPACE_ROOT is None:
    raise FileNotFoundError("Could not locate the Inception workspace from the notebook directory.")
TEMPORAL_TESTING_DIR = WORKSPACE_ROOT / "Temporal-Testing"
if str(TEMPORAL_TESTING_DIR) not in sys.path:
    sys.path.insert(0, str(TEMPORAL_TESTING_DIR))

from reproducibility import DEFAULT_SEED, seed_everything
SEED = DEFAULT_SEED
seed_everything(SEED)

42

In [3]:
with (WORKSPACE_ROOT / "Fin-RATE" / "qa" / "LT-QA.json").open("r", encoding="utf-8") as file:
    ltqa = json.load(file)

In [4]:
nqa = 100
question_sampler = random.Random(SEED)
smoke_test_ltqa = question_sampler.sample(ltqa, nqa)
sample_docs = set()
for sample_qa in smoke_test_ltqa:
    docs = set(sample_qa["doc_ids"])
    sample_docs.update(docs)

subset_questions_path = TEMPORAL_TESTING_DIR / "ltqa_subset_questions.json"
subset_questions_path.write_text(
    json.dumps(smoke_test_ltqa, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

subset_doc_ids_path = TEMPORAL_TESTING_DIR / "ltqa_subset_doc_ids.json"
subset_doc_ids_path.write_text(
    json.dumps(sorted(sample_docs), ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print(f"Saved {len(smoke_test_ltqa)} LT-QA records to {subset_questions_path}")
print(f"Saved {len(sample_docs)} connected document IDs to {subset_doc_ids_path}")

Saved 100 LT-QA records to C:\Users\hello2\Documents\Inception\Temporal-Testing\ltqa_subset_questions.json
Saved 218 connected document IDs to C:\Users\hello2\Documents\Inception\Temporal-Testing\ltqa_subset_doc_ids.json


In [5]:
list(sample_docs)[2]

'doc_013681'

In [6]:
from ChromaSetup import build_chroma_database as build_plain_chroma
from ChromaSetupMetaData import (
    build_chroma_database as build_metadata_chroma,
    create_embedding_function,
)

embedding_function = create_embedding_function(
    backend="ollama",
    model_name="embeddinggemma:latest",
    ollama_embedding_batch_size=32,
    seed=SEED,
)

plain_stats = build_plain_chroma(
    document_ids=sorted(sample_docs),
    db_dir=WORKSPACE_ROOT / "Fin-RATE" / "chroma_db_ltqa_subset",
    collection_name="fin_rate_ltqa_subset",
    embedding_function=embedding_function,
    batch_size=256,
    seed=SEED,
)

metadata_stats = build_metadata_chroma(
    document_ids=sorted(sample_docs),
    db_dir=WORKSPACE_ROOT / "Fin-RATE" / "chroma_metadata_db_ltqa_subset",
    collection_name="fin_rate_ltqa_subset",
    embedding_function=embedding_function,
    seed=SEED,
)

print("Plain Chroma:", plain_stats["build_seconds"], "seconds")
print("Metadata Chroma:", metadata_stats["build_seconds"], "seconds")

Plain Chroma: 555.4843871999765 seconds
Metadata Chroma: 553.6129819999915 seconds


In [10]:
import subprocess
import sys

In [12]:
benchmark_command = [
    sys.executable,
    str(TEMPORAL_TESTING_DIR / "benchmark_retrievers.py"),
    "--systems", "chroma,chroma_metadata",
    "--qa-file", str(TEMPORAL_TESTING_DIR / "ltqa_subset_questions.json"),
    "--chroma-db-dir", str(WORKSPACE_ROOT / "Fin-RATE" / "chroma_db_ltqa_subset"),
    "--chroma-metadata-db-dir", str(
        WORKSPACE_ROOT / "Fin-RATE" / "chroma_metadata_db_ltqa_subset"
    ),
    "--chroma-collection", "fin_rate_ltqa_subset",
    "--embedding-backend", "ollama",
    "--embedding-model", "embeddinggemma:latest",
    "--metadata-filter-mode", "ollama",
    "--metadata-llm-model", "qwen3:4b",
    "--ollama-url", "http://127.0.0.1:11434",
    "--warmup", "0",
]

subprocess.run(
    benchmark_command,
    cwd=TEMPORAL_TESTING_DIR,
    check=True,
)

CompletedProcess(args=['c:\\Users\\hello2\\AppData\\Local\\Programs\\Python\\Python312\\python.exe', 'C:\\Users\\hello2\\Documents\\Inception\\Temporal-Testing\\benchmark_retrievers.py', '--systems', 'chroma,chroma_metadata', '--qa-file', 'C:\\Users\\hello2\\Documents\\Inception\\Temporal-Testing\\ltqa_subset_questions.json', '--chroma-db-dir', 'C:\\Users\\hello2\\Documents\\Inception\\Fin-RATE\\chroma_db_ltqa_subset', '--chroma-metadata-db-dir', 'C:\\Users\\hello2\\Documents\\Inception\\Fin-RATE\\chroma_metadata_db_ltqa_subset', '--chroma-collection', 'fin_rate_ltqa_subset', '--embedding-backend', 'ollama', '--embedding-model', 'embeddinggemma:latest', '--metadata-filter-mode', 'ollama', '--metadata-llm-model', 'qwen3:4b', '--ollama-url', 'http://127.0.0.1:11434', '--warmup', '0'], returncode=0)

In [ ]:
from Engram_Temporal import (
    OllamaEmbedder,
    OllamaLLM,
    _service_namespace_dir,
    build_store,
)

temporal_namespace = "fin-rate-ltqa-subset"
temporal_data_dir = WORKSPACE_ROOT / "Fin-RATE" / "engram_data_ltqa_subset"
temporal_store_dir = _service_namespace_dir(
    temporal_data_dir,
    temporal_namespace,
)

llm = OllamaLLM(
    "qwen3:4b",
    base_url="http://127.0.0.1:11434",
    timeout=300,
    num_ctx=4096,
    num_predict=768,
    seed=SEED,
)

embedder = OllamaEmbedder(
    "embeddinggemma:latest",
    base_url="http://127.0.0.1:11434",
    timeout=300,
)


In [ ]:

engram_stats = build_store(
    corpus_path=WORKSPACE_ROOT / "Fin-RATE" / "corpus" / "corpus" / "corpus.jsonl",
    store_dir=temporal_store_dir,
    namespace=temporal_namespace,
    document_ids=sorted(sample_docs),
    limit=None,
    chunk_size_tokens=512,
    chunk_overlap_tokens=64,
    reset=True,

    # Deterministic chunk summaries; avoids one Qwen call per chunk.
    summarize=True,
    llm_summaries=False,

    # Temporal enrichment and broad-question navigation.
    llm_document_metadata=True,
    filing_summaries=True,
    llm_filing_summaries=True,

    # Current temporal query path retrieves raw chunks, not extracted text facts.
    extract_text_facts=False,
    deep_relationships=True,

    llm=llm,
    embedder=embedder,
    metadata_llm_text_chars=4000,
    filing_summary_input_chars=12000,
    seed=SEED,
)

print(engram_stats)
print("Temporal Engram store:", temporal_store_dir)

#WILL TAKE ABOUT 90 to 150 minutes to RUN


RuntimeError: Could not reach Ollama at http://127.0.0.1:11434. Start Ollama and check the URL.

In [6]:
from ChromaSetupMetaData import build_chroma_database,create_embedding_function
embd = create_embedding_function(backend="ollama",model_name="embeddinggemma",device="cuda")


In [6]:
build_chroma_database(
    document_ids=list(sample_docs),
    collection_name="fin_rate_mini",
    embedding_function=embd
)

{'corpus_path': 'C:\\Users\\hello2\\Documents\\Inception\\Fin-RATE\\corpus\\corpus\\corpus.jsonl',
 'database_dir': 'C:\\Users\\hello2\\Documents\\Inception\\Fin-RATE\\chroma_db',
 'collection_name': 'fin_rate_mini',
 'records_read': 15039,
 'documents_seen': 125,
 'document_ids_filter': ['doc_000026',
  'doc_000176',
  'doc_000266',
  'doc_000503',
  'doc_000586',
  'doc_000915',
  'doc_000919',
  'doc_000922',
  'doc_001027',
  'doc_001033',
  'doc_001049',
  'doc_001051',
  'doc_001055',
  'doc_001716',
  'doc_001734',
  'doc_001747',
  'doc_001748',
  'doc_001749',
  'doc_001781',
  'doc_001783',
  'doc_001791',
  'doc_001793',
  'doc_001833',
  'doc_001839',
  'doc_001840',
  'doc_001856',
  'doc_001859',
  'doc_001861',
  'doc_002597',
  'doc_002637',
  'doc_004252',
  'doc_004253',
  'doc_004255',
  'doc_004437',
  'doc_004709',
  'doc_004720',
  'doc_004783',
  'doc_005376',
  'doc_005463',
  'doc_005539',
  'doc_005640',
  'doc_005642',
  'doc_005658',
  'doc_005707',
  'doc_0

In [7]:
smoke_test_ltqa[1]["question"]

"What progress has BeiGene's cornerstone product BRUKINSA made in global market coverage from 2022 to 2023? How has the description of its leadership position in the Chinese BTK inhibitor market changed?"

In [8]:
from ChromaSetupMetaData import answer_question
result = answer_question(
    question=smoke_test_ltqa[1]["question"],
    collection_name="fin_rate_mini",
    embedding_function=embd,
    n_results=5,
    llm_model="qwen3:4b",
)

In [26]:
smoke_test_ltqa[1]["doc_ids"]

['doc_013510', 'doc_013681']

In [ ]:
[chun["id"] for chun in result["chunks"]]

['doc_013510::chunk_0002',
 'doc_013593::chunk_0003',
 'doc_013510::chunk_0014',
 'doc_013794::chunk_0063',
 'doc_013681::chunk_0006']

In [24]:
[docid[:10] for docid in [chun["id"] for chun in result["chunks"]]]

['doc_013510', 'doc_013593', 'doc_013510', 'doc_013794', 'doc_013681']

In [18]:
print(smoke_test_ltqa[1]["answer"])
print("\n___________________________________________\n")
print(result["answer"])

In terms of market coverage, BRUKINSA was already 'approved in >65 markets' in 2022. The 2023 summary continued the statement of being 'approved in more than 65 markets,' indicating that its global coverage network has reached a stable and broad stage in terms of numbers, with the focus potentially shifting from开拓新市场 to deepening penetration and growth in already approved markets. Regarding the description of its leadership position in the Chinese market, the 2022 summary explicitly stated that BRUKINSA was the 'market leader in China for BTKi class.' The 2023 summary, while not repeating the exact same phrase, emphasized that BRUKINSA is the 'first and only BTK inhibitor to demonstrate superior efficacy versus ibrutinib in R/R CLL,' which is the key clinical cornerstone of its market leadership. Combined with its remarkable sales growth of 128.5% in 2023, it can be inferred that its market leadership position has been consolidated and strengthened.

___________________________________

In [5]:
len(ltqa)

2500

In [6]:
#System 1: Normal Vector Retrieval
from vector_retrieval import retrieval_pipeline
recalls = []
mrr = 0
for qa in smoke_test_ltqa:
    chunks = retrieval_pipeline(qa["question"], seed=SEED)
    golden = qa["doc_ids"]
    #Recall 5
    recall5 = len(set(list(set(chunks))[:5]) & set(golden))/len(golden)
    recall10 = len(set(list(set(chunks))[:10]) & set(golden))/len(golden)

    recalls.append((recall5,recall10))

    rr = 0
    for rank, item in enumerate(chunks, start=1):
        if item in golden:
            rr = 1 / rank
            break
    mrr += rr

#Combine chunks of same docs and then calc recall

print(np.mean([recs[0] for recs in recalls]))
print(np.mean([recs[1] for recs in recalls]))
#print(mrr/nqa)


0.0675
0.14116666666666666


In [7]:
#System 2: ChromaDB Vector Retrieval
from ChromaSetup import create_embedding_function, build_chroma_database, retrieve_relevant_chunks



In [ ]:
!{sys.executable} "{TEMPORAL_TESTING_DIR / 'ChromaSetup.py'}" --build --embedding-backend bge --device cuda --seed {SEED}

In [10]:
recalls = []
for qa in smoke_test_ltqa:
    chunks = retrieve_relevant_chunks(qa["question"], seed=SEED)
    golden = qa["doc_ids"]
    #Recall 5
    recall5 = len(set(list(set([chun['metadata']['doc_id'] for chun in chunks]))[:5]) & set(golden))/len(golden)
    recall10 = len(set(list(set([chun['metadata']['doc_id'] for chun in chunks]))[:10]) & set(golden))/len(golden)
    recalls.append((recall5,recall10))

print(np.mean([recs[0] for recs in recalls]))
print(np.mean([recs[1] for recs in recalls]))

0.0
0.0


In [28]:
#META DATA
recalls = []
for qa in smoke_test_ltqa[:10]:
    result = answer_question(
    question=qa["question"],
    collection_name="fin_rate_mini",
    embedding_function=embd,
    n_results=5,
    llm_model="qwen3:4b",
)
    chunks = [docid[:10] for docid in [chun["id"] for chun in result["chunks"]]]
    golden = qa["doc_ids"]
    #Recall 5
    recall5 = len(set(chunks[:5]) & set(golden))/len(golden)
    recall10 = len(set(chunks[:10]) & set(golden))/len(golden)
    recalls.append((recall5,recall10))

print(np.mean([recs[0] for recs in recalls]))
print(np.mean([recs[1] for recs in recalls]))

0.7035714285714285
0.7035714285714285


In [ ]:
from sentence_transformers import SentenceTransformer
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer("BAAI/bge-m3", device=device)

embeddings = model.encode(
    ["What is reranking in RAG?"],
    batch_size=8,
    normalize_embeddings=True,
)
print(embeddings.shape)

In [16]:
#System 3: CRAG 
'''

from ChromaSetup import create_embedding_function, build_chroma_database, retrieve_relevant_chunks

ef = create_embedding_function(
    backend="sentence-transformers",
    device="cuda",
    seed=SEED,
)

recalls = []
for qa in smoke_test_ltqa:
    chunks = retrieve_relevant_chunks(qa["question"], seed=SEED)
    golden = qa["doc_ids"]
    #Recall 5
    recall5 = len(set([chun['metadata']['doc_id'] for chun in chunks][:5]) & set(golden))/len(golden)
    recall10 = len(set([chun['metadata']['doc_id'] for chun in chunks][:10]) & set(golden))/len(golden)
    recalls.append((recall5,recall10))

print(np.mean([recs[0] for recs in recalls]))
print(np.mean([recs[1] for recs in recalls]))

'''

'\n\nfrom ChromaSetup import create_embedding_function, build_chroma_database, retrieve_relevant_chunks\n\nef = create_embedding_function(\n    backend="sentence-transformers",\n    device="cuda",\n)\n\nrecalls = []\nfor qa in smoke_test_ltqa:\n    chunks = retrieve_relevant_chunks(qa["question"])\n    golden = qa["doc_ids"]\n    #Recall 5\n    recall5 = len(set([chun[\'metadata\'][\'doc_id\'] for chun in chunks][:5]) & set(golden))/len(golden)\n    recall10 = len(set([chun[\'metadata\'][\'doc_id\'] for chun in chunks][:10]) & set(golden))/len(golden)\n    recalls.append((recall5,recall10))\n\nprint(np.mean([recs[0] for recs in recalls]))\nprint(np.mean([recs[1] for recs in recalls]))\n\n'